--- Day 8: Haunted Wasteland ---
You're still riding a camel across Desert Island when you spot a sandstorm quickly approaching. When you turn to warn the Elf, she disappears before your eyes! To be fair, she had just finished warning you about ghosts a few minutes ago.

One of the camel's pouches is labeled "maps" - sure enough, it's full of documents (your puzzle input) about how to navigate the desert. At least, you're pretty sure that's what they are; one of the documents contains a list of left/right instructions, and the rest of the documents seem to describe some kind of network of labeled nodes.

It seems like you're meant to use the left/right instructions to navigate the network. Perhaps if you have the camel follow the same instructions, you can escape the haunted wasteland!

After examining the maps for a bit, two nodes stick out: AAA and ZZZ. You feel like AAA is where you are now, and you have to follow the left/right instructions until you reach ZZZ.

This format defines each node of the network individually. For example:

RL

AAA = (BBB, CCC)
BBB = (DDD, EEE)
CCC = (ZZZ, GGG)
DDD = (DDD, DDD)
EEE = (EEE, EEE)
GGG = (GGG, GGG)
ZZZ = (ZZZ, ZZZ)
Starting with AAA, you need to look up the next element based on the next left/right instruction in your input. In this example, start with AAA and go right (R) by choosing the right element of AAA, CCC. Then, L means to choose the left element of CCC, ZZZ. By following the left/right instructions, you reach ZZZ in 2 steps.

Of course, you might not find ZZZ right away. If you run out of left/right instructions, repeat the whole sequence of instructions as necessary: RL really means RLRLRLRLRLRLRLRL... and so on. For example, here is a situation that takes 6 steps to reach ZZZ:

LLR

AAA = (BBB, BBB)
BBB = (AAA, ZZZ)
ZZZ = (ZZZ, ZZZ)
Starting at AAA, follow the left/right instructions. How many steps are required to reach ZZZ?



In [32]:
with open('input08.txt') as file:
    data = file.read()
instr_init = [0 if lr == 'L' else 1 for lr in data.splitlines()[0]]

nodes = {}
for line in data.splitlines()[2:]:
    source = line[:3]
    left = line.split('(')[1][:3]
    right = line.split(')')[0][-3:]
    nodes[source] = (left, right)
instr_init[:5], next(iter(nodes.items()))

([0, 1, 1, 0, 1], ('RBX', ('TMF', 'KTP')))

In [33]:
from itertools import cycle
instr = cycle(instr_init)

pos = 'AAA'
end = 'ZZZ'
for path_len, dir in enumerate(instr, start=1):
    pos = nodes[pos][dir]
    if pos == end:
        break
pos, path_len
    


('ZZZ', 21389)

Your puzzle answer was 21389.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The sandstorm is upon you and you aren't any closer to escaping the wasteland. You had the camel follow the instructions, but you've barely left your starting position. It's going to take significantly more steps to escape!

What if the map isn't for people - what if the map is for ghosts? Are ghosts even bound by the laws of spacetime? Only one way to find out.

After examining the maps a bit longer, your attention is drawn to a curious fact: the number of nodes with names ending in A is equal to the number ending in Z! If you were a ghost, you'd probably just start at every node that ends with A and follow all of the paths at the same time until they all simultaneously end up at nodes that end with Z.

For example:

LR

11A = (11B, XXX)
11B = (XXX, 11Z)
11Z = (11B, XXX)
22A = (22B, XXX)
22B = (22C, 22C)
22C = (22Z, 22Z)
22Z = (22B, 22B)
XXX = (XXX, XXX)
Here, there are two starting nodes, 11A and 22A (because they both end with A). As you follow each left/right instruction, use that instruction to simultaneously navigate away from both nodes you're currently on. Repeat this process until all of the nodes you're currently on end with Z. (If only some of the nodes you're on end with Z, they act like any other node and you continue as normal.) In this example, you would proceed as follows:

Step 0: You are at 11A and 22A.
Step 1: You choose all of the left paths, leading you to 11B and 22B.
Step 2: You choose all of the right paths, leading you to 11Z and 22C.
Step 3: You choose all of the left paths, leading you to 11B and 22Z.
Step 4: You choose all of the right paths, leading you to 11Z and 22B.
Step 5: You choose all of the left paths, leading you to 11B and 22C.
Step 6: You choose all of the right paths, leading you to 11Z and 22Z.
So, in this example, you end up entirely on nodes that end in Z after 6 steps.

Simultaneously start on every node that ends with A. How many steps does it take before you're only on nodes that end with Z?

In [34]:
# # Sample validation:
# instr = cycle((0,1))

# data = """11A = (11B, XXX)
# 11B = (XXX, 11Z)
# 11Z = (11B, XXX)
# 22A = (22B, XXX)
# 22B = (22C, 22C)
# 22C = (22Z, 22Z)
# 22Z = (22B, 22B)
# XXX = (XXX, XXX)"""
# nodes = {}
# for line in data.splitlines():
#     source = line[:3]
#     left = line.split('(')[1][:3]
#     right = line.split(')')[0][-3:]
#     nodes[source] = (left, right)
# next(iter(nodes.items()))

# pos = [node for node in nodes if node.endswith('A')]
# for path_len, dir in enumerate(instr, start=1):
#     pos = [nodes[p][dir] for p in pos]
#     if all(p.endswith('Z') for p in pos):
#         break
# pos, path_len

In [38]:
# Let's see if bruteforcing suffices
start_pos = [node for node in nodes if node.endswith('A')]
start_pos

['DVA', 'MPA', 'TDA', 'AAA', 'FJA', 'XPA']

In [ ]:
instr = cycle(instr_init)
pos = start_pos.copy()
for p in start_pos:
    for path_len, dir in enumerate(instr, start=1):
        pos = [nodes[p][dir] for p in pos]
        if all(p.endswith('Z') for p in pos):
            break
pos, path_len

KeyboardInterrupt: 

In [39]:
# Ok this seems slow. Let us instead:
# Calculate for each A how many steps until it reaches its first Z (and which)
# Calculate for each Z how many steps until it reaches another Z
# I am sure there is a nice mathematical solution for this but let's do it this way

end_pos = []
for pos in start_pos:
    instr = cycle(instr_init)
    for path_len, dir in enumerate(instr, start=1):
        pos = nodes[pos][dir]
        if pos.endswith('Z'):
            end_pos.append((pos, path_len))
            break
start_pos, end_pos

(['DVA', 'MPA', 'TDA', 'AAA', 'FJA', 'XPA'],
 [('MSZ', 23147),
  ('DGZ', 19631),
  ('MVZ', 12599),
  ('ZZZ', 21389),
  ('CJZ', 17873),
  ('QFZ', 20803)])

In [40]:
z_cycles = []
for pos, _ in end_pos:
    z_start = pos
    instr = cycle(instr_init)
    for path_len, dir in enumerate(instr, start=1):
        pos = nodes[pos][dir]
        if pos.endswith('Z'):
            z_cycles.append((z_start, pos, path_len))
            break
z_cycles

[('MSZ', 'MSZ', 23147),
 ('DGZ', 'DGZ', 19631),
 ('MVZ', 'MVZ', 12599),
 ('ZZZ', 'ZZZ', 21389),
 ('CJZ', 'CJZ', 17873),
 ('QFZ', 'QFZ', 20803)]

In [42]:
# Ha what a neat (and unlikely) match: The time it takes to reach Z from A is exactly the same as the time needed to cycle from Z back to (the identical) Z
from math import lcm
lcm(*[x[2] for x in z_cycles])

21083806112641

That's the right answer! You are one gold star closer to restoring snow operations.

You have completed Day 8! You can [Share] this victory or [Return to Your Advent Calendar].